# Debouncing SRCLK: a simulation

A pushbutton ties the 74HC595's SRCLK to 3.3 V; a 10 kΩ pull-down returns it to 0 V. This notebook simulates the SRCLK voltage for one bouncy press and release, without and with a 100 nF capacitor across the pull-down, and counts the rising edges the chip sees.

The bounce pattern is invented, and stretched so it shows at a millisecond scale; it is not a measurement. The last cell writes `../docs/data/debounce.dat`, which the Stage 0c chapter plots with pgfplots.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

VCC = 3.3                        # V
TAU = 10e3 * 100e-9 * 1e3        # ms: 10 kΩ × 100 nF
VIL, VIH = 0.3 * VCC, 0.7 * VCC  # 74HC input levels: below VIL reads LOW, above VIH reads HIGH
DT = 0.001                       # ms: simulation step, 1 µs
t = np.arange(0, 7, DT)          # ms

## Contacts

Intervals (ms) during which the contacts are closed: a press at 1 ms bouncing three times, and a release at 4 ms bouncing twice.

In [ ]:
CLOSED = [(1.00, 1.15), (1.25, 1.35), (1.45, 1.70), (1.80, 4.00), (4.12, 4.20), (4.30, 4.36)]
closed = np.any([(a <= t) & (t < b) for a, b in CLOSED], axis=0)

## SRCLK voltage

- Closed: the button ties SRCLK to 3.3 V. With the capacitor, it charges at once, through no resistance.
- Open: the pull-down takes SRCLK to 0 V, at once without the capacitor; with it, the capacitor drains through 10 kΩ, $v \leftarrow v\, e^{-\Delta t/\tau}$ per step.

In [ ]:
v_bare = np.where(closed, VCC, 0.0)

v_cap = np.zeros_like(t)
for i in range(1, len(t)):
    v_cap[i] = VCC if closed[i] else v_cap[i - 1] * np.exp(-DT / TAU)

## Rising edges

The chip sees a rising edge when the input passes VIH, having last been below VIL. Each one is a shift.

In [ ]:
def rising_edges(v):
    n, low = 0, True
    for x in v:
        if low and x > VIH:
            n, low = n + 1, False
        elif x < VIL:
            low = True
    return n

print("pull-down only:", rising_edges(v_bare), "rising edges")
print("with capacitor:", rising_edges(v_cap), "rising edges")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
for ax, v, title in zip(axes, [v_bare, v_cap], ["Pull-down only", "Capacitor across the pull-down"]):
    ax.axhspan(VIL, VIH, color="0.9")
    ax.plot(t, v)
    ax.set_title(title, loc="left")
axes[1].set_xlabel("time (ms)");

## Data for the chapter

Every 10th sample (10 µs apart) is plenty for the plot: columns `t` (ms), `bare` and `cap` (V).

In [ ]:
out = Path("../docs/data/debounce.dat")
out.parent.mkdir(exist_ok=True)
np.savetxt(out, np.column_stack([t, v_bare, v_cap])[::10], fmt="%.3f",
           header="t bare cap", comments="")
out